# NYC Yellow Taxi Trip Analysis

## Project Overview

This project aims to perform an end-to-end data science workflow on the NYC Yellow Taxi Trip dataset. The analysis includes data understanding, data cleaning, exploratory data analysis (EDA), feature engineering, preprocessing, machine learning modeling, and model evaluation.

The dataset used in this project is publicly provided by the **New York City Taxi and Limousine Commission (TLC)**.

> **Dataset Scope:** Only the **January 2026** Yellow Taxi Trip Records are used in this project. This subset contains millions of real-world taxi trips and is sufficient for comprehensive analysis while remaining computationally manageable on a local machine.

> Data source : https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 100
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

### Load the dataset

In [4]:
df = pd.read_parquet('data/yellow_tripdata_2026-01.parquet')
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.20,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.90,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.70,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.70,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.50,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,NaN,237,161,0,17.09,0.00,0.5,0.00,0.0,1.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,NaN,42,42,0,23.19,0.00,0.5,0.00,0.0,1.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,NaN,137,69,0,29.21,0.00,0.5,0.00,0.0,1.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,NaN,137,162,0,19.19,0.00,0.5,0.00,0.0,1.0,23.94,NaN,NaN,0.75


In [5]:
taxi_zone_lookup = pd.read_csv('data/taxi_zone_lookup.csv')
taxi_zone_lookup

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


## Chapter 1 : Data Overview

In [6]:
file_path = "data/yellow_tripdata_2026-01.parquet"

trip_duration = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

overview_df = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Features",
        "Pickup Date Range",
        "Total Observation Days",
        "Dataset Size (MB)",
        "Exact Duplicate Records",
        "Total Missing Cells",
        "Rows With Missing Values",
        "Negative Fare Records",
        "Non-positive Duration Records",
    ],
    "Value": [
        f"{len(df):,}",
        df.shape[1],
        f"{df['tpep_pickup_datetime'].min():%Y-%m-%d} to "
        f"{df['tpep_pickup_datetime'].max():%Y-%m-%d}",
        df["tpep_pickup_datetime"].dt.date.nunique(),
        round(os.path.getsize(file_path) / (1024 ** 2), 2),
        f"{df.duplicated().sum():,}",
        f"{df.isna().sum().sum():,}",
        f"{df.isna().any(axis=1).sum():,}",
        f"{(df['fare_amount'] < 0).sum():,}",
        f"{(trip_duration <= 0).sum():,}",
    ],
})

display(overview_df)

,Metric,Value
0,Total Records,"3,724,889"
1,Total Features,20
2,Pickup Date Range,2025-12-31 to 2026-02-01
3,Total Observation Days,33
4,Dataset Size (MB),61.19
5,Exact Duplicate Records,0
6,Total Missing Cells,"5,440,290"
7,Rows With Missing Values,"1,088,058"
8,Negative Fare Records,"39,463"
9,Non-positive Duration Records,"45,070"


## Chapter 2 : Data Quality Check

In [7]:
missing_summary = (
    df.isna()
      .sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda data: (data["missing_count"] / len(df) * 100).round(2))
      .query("missing_count > 0")
      .sort_values("missing_count", ascending=False)
)

display(missing_summary)

,missing_count,missing_pct
passenger_count,1088058,29.21
RatecodeID,1088058,29.21
store_and_fwd_flag,1088058,29.21
congestion_surcharge,1088058,29.21
Airport_fee,1088058,29.21


In [8]:
df_clean = df.copy()

df_clean["trip_duration_min"] = (
    df_clean["tpep_dropoff_datetime"] - df_clean["tpep_pickup_datetime"]
).dt.total_seconds() / 60

df_clean = df_clean[
    (df_clean["trip_duration_min"] > 0) &
    (df_clean["fare_amount"] >= 0) &
    (df_clean["total_amount"] >= 0)
].copy()

print(f"Before cleaning: {len(df):,}")
print(f"After cleaning:  {len(df_clean):,}")
print(f"Rows removed:    {len(df) - len(df_clean):,}")

Before cleaning: 3,724,889
After cleaning:  3,639,817
Rows removed:    85,072


### Feature Engineering

In [9]:
pickup_datetime = df_clean["tpep_pickup_datetime"]

# Time-based features
# Keep trip_duration_min from the cleaning step because it is useful for EDA and ML.
df_clean["pickup_date"] = pickup_datetime.dt.date
df_clean["pickup_hour"] = pickup_datetime.dt.hour
df_clean["pickup_dayofweek"] = pickup_datetime.dt.dayofweek
df_clean["pickup_day_name"] = pickup_datetime.dt.day_name()
df_clean["is_weekend"] = (df_clean["pickup_dayofweek"] >= 5).astype(int)
df_clean["pickup_period"] = pd.cut(
    df_clean["pickup_hour"],
    bins=[-1, 5, 10, 15, 19, 23],
    labels=["Late Night", "Morning", "Afternoon", "Evening", "Night"]
)
df_clean["is_rush_hour"] = df_clean["pickup_hour"].isin([7, 8, 9, 16, 17, 18, 19]).astype(int)
df_clean["is_night"] = df_clean["pickup_hour"].isin([0, 1, 2, 3, 4, 5]).astype(int)

# EDA features
df_clean["speed_mph"] = np.where(
    df_clean["trip_duration_min"] > 0,
    df_clean["trip_distance"] / (df_clean["trip_duration_min"] / 60),
    np.nan
)
df_clean["tip_percentage"] = np.where(
    df_clean["fare_amount"] > 0,
    df_clean["tip_amount"] / df_clean["fare_amount"] * 100,
    np.nan
)
df_clean["fare_per_mile"] = np.where(
    df_clean["trip_distance"] > 0,
    df_clean["fare_amount"] / df_clean["trip_distance"],
    np.nan
)
df_clean["distance_category"] = pd.cut(
    df_clean["trip_distance"],
    bins=[-0.01, 1, 3, 7, 15, np.inf],
    labels=["Very Short", "Short", "Medium", "Long", "Very Long"]
)

df_clean["route_id"] = (
    df_clean["PULocationID"].astype(str)
    + " -> "
    + df_clean["DOLocationID"].astype(str)
)

df_clean.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,pickup_day_name,is_weekend,pickup_period,is_rush_hour,is_night,speed_mph,tip_percentage,fare_per_mile,distance_category,route_id
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,Thursday,0,Late Night,0,1,10.486486,50.833333,7.422680,Very Short,239 -> 238
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,...,Thursday,0,Late Night,0,1,9.446064,0.000000,8.777778,Very Short,163 -> 162
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,...,Thursday,0,Late Night,0,1,9.455910,23.364486,7.642857,Short,43 -> 237
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,Thursday,0,Late Night,0,1,7.822430,28.708010,6.935484,Medium,142 -> 209
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,...,Thursday,0,Late Night,0,1,9.600000,28.518519,6.250000,Short,88 -> 144


In [10]:
pickup_zones = taxi_zone_lookup.rename(columns={
    "LocationID": "PULocationID",
    "Borough": "pickup_borough",
    "Zone": "pickup_zone",
    "service_zone": "pickup_service_zone"
})[["PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone"]]

dropoff_zones = taxi_zone_lookup.rename(columns={
    "LocationID": "DOLocationID",
    "Borough": "dropoff_borough",
    "Zone": "dropoff_zone",
    "service_zone": "dropoff_service_zone"
})[["DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone"]]

df_enriched = (
    df_clean
    .merge(pickup_zones, on="PULocationID", how="left", validate="many_to_one")
    .merge(dropoff_zones, on="DOLocationID", how="left", validate="many_to_one")
)

zone_label_cols = [
    "pickup_borough", "pickup_zone", "pickup_service_zone",
    "dropoff_borough", "dropoff_zone", "dropoff_service_zone"
]
df_enriched[zone_label_cols] = df_enriched[zone_label_cols].fillna("Unknown")

df_enriched["borough_route"] = (
    df_enriched["pickup_borough"]
    + " -> "
    + df_enriched["dropoff_borough"]
)
df_enriched["zone_route"] = (
    df_enriched["pickup_zone"]
    + " -> "
    + df_enriched["dropoff_zone"]
)
df_enriched["is_interborough_trip"] = (
    df_enriched["pickup_borough"] != df_enriched["dropoff_borough"]
).astype(int)

print(f"Rows before join: {len(df_clean):,}")
print(f"Rows after join:  {len(df_enriched):,}")
print()
print("Unknown pickup zones:", (df_enriched["pickup_zone"] == "Unknown").sum())
print("Unknown drop-off zones:", (df_enriched["dropoff_zone"] == "Unknown").sum())

df_enriched[[
    "PULocationID", "pickup_borough", "pickup_zone", "pickup_service_zone",
    "DOLocationID", "dropoff_borough", "dropoff_zone", "dropoff_service_zone",
    "borough_route", "zone_route", "is_interborough_trip"
]].head()

Rows before join: 3,639,817
Rows after join:  3,639,817

Unknown pickup zones: 4302
Unknown drop-off zones: 5665


,PULocationID,pickup_borough,pickup_zone,pickup_service_zone,DOLocationID,dropoff_borough,dropoff_zone,dropoff_service_zone,borough_route,zone_route,is_interborough_trip
0,239,Manhattan,Upper West Side South,Yellow Zone,238,Manhattan,Upper West Side North,Yellow Zone,Manhattan -> Manhattan,Upper West Side South -> Upper West Side North,0
1,163,Manhattan,Midtown North,Yellow Zone,162,Manhattan,Midtown East,Yellow Zone,Manhattan -> Manhattan,Midtown North -> Midtown East,0
2,43,Manhattan,Central Park,Yellow Zone,237,Manhattan,Upper East Side South,Yellow Zone,Manhattan -> Manhattan,Central Park -> Upper East Side South,0
3,142,Manhattan,Lincoln Square East,Yellow Zone,209,Manhattan,Seaport,Yellow Zone,Manhattan -> Manhattan,Lincoln Square East -> Seaport,0
4,88,Manhattan,Financial District South,Yellow Zone,144,Manhattan,Little Italy/NoLiTa,Yellow Zone,Manhattan -> Manhattan,Financial District South -> Little Italy/NoLiTa,0


In [ ]:
numerical_cols = [
    "passenger_count",
    "trip_distance",
    "trip_duration_min",
    "fare_amount",
    "extra",
    "mta_tax",
    "tip_amount",
    "tolls_amount",
    "improvement_surcharge",
    "total_amount",
    "congestion_surcharge",
    "Airport_fee",
    "cbd_congestion_fee",
    "speed_mph",
    "tip_percentage",
    "fare_per_mile"
]

def outlier_summary(data, numerical_cols) :

    results = []

    for col in numerical_cols :
        series = data[col]
        q1 = series.quantile(.25)
        q3 = series.quantile(.75)

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier = (
            (series < lower) |
            (series > upper)
        ).sum()

        results.append({
            "column" : col,
            "count" : len(series),
            "mean" : round(series.mean(), 2),
            "median" : round(series.median(), 2),
            "min" : round(series.min(), 2),
            "max" : round(series.max(), 2),
            "iqr_lower_bound": round(lower,2),
            "iqr_upper_bound": round(upper,2),
            "outlier_count": outlier,
            "outlier_pct": round(outlier/len(series)*100,2)
        })

    return (
        pd.DataFrame(results).sort_values("outlier_pct", ascending=False)
    )

df_outlier = outlier_summary(df_enriched, numerical_cols)
df_outlier

,column,count,mean,median,min,max,iqr_lower_bound,iqr_upper_bound,outlier_count,outlier_pct
0,passenger_count,3639817,1.26,1.00,0.00,9.00,1.00,1.00,468723,12.88
1,trip_distance,3639817,6.53,1.82,0.00,269097.48,-3.12,7.88,409047,11.24
9,total_amount,3639817,29.88,23.22,0.00,2560.20,-7.88,59.12,307084,8.44
10,congestion_surcharge,3639817,2.21,2.50,0.00,2.50,2.50,2.50,291423,8.01
13,speed_mph,3639817,24.38,9.25,0.00,2218899.96,-2.52,22.19,258362,7.10
3,fare_amount,3639817,21.35,15.60,0.00,2555.20,-14.30,50.50,242930,6.67
7,tolls_amount,3639817,0.51,0.00,0.00,122.22,0.00,0.00,235378,6.47
11,Airport_fee,3639817,0.15,0.00,0.00,1.75,0.00,0.00,225769,6.20
15,fare_per_mile,3639817,27.48,7.54,0.00,49000.00,-0.61,16.27,212485,5.84
2,trip_duration_min,3639817,17.42,13.53,0.02,7507.90,-11.32,40.95,205155,5.64
